# 31 - The trade-off curve: NGC 7000, the night of 2026-09-05

**Purpose.** Draw MISSION's deliverable for the one target and night where both halves of the
model have now passed: the **Pareto curve** between the SNR of faint extended signal and star
colour. `27` showed the SNR half ranks settings correctly on this night; `29` showed the colour
half predicts which stars clip. This notebook puts them together and nothing else.

**What it computes.** For every setting - a sub length and a gain - two numbers:

1. **SNR of faint signal over the whole night**, as a share of the best any setting reaches.
   Fixed wall clock, not fixed integration: the night is the night, and every sub costs `t_dead`.
2. **The share of `29`'s 8033 stars whose core clips** at that setting.

Then it keeps the settings no other setting beats on both. That line is the curve. **Choosing a
point on it is not this notebook's job**; MISSION leaves that to Denis, per target.

**The settings.** Sub length from 10 to 600 s. Gain **50, 100 and 200 only**: those are the
gains where `linear_to_at_least` - the ceiling - was proved, and the model refuses to interpolate a
ceiling between rungs. That is a limit of the record, not a choice; gain 50 to 200 straddles HCG,
which is the comparison MISSION's *gain nearly cancels* is about.

**Two choices, agreed before this notebook was written.**

- **No stacking loss: `eta_comb = 1`.** The model's own table is the bias ladder, which falls to
  0.54 by N = 128 and refuses beyond it. Registered, dithered lights measured 0.96-1.02 up to
  N = 18 (`25`), and the bias ladder's fall is a fixed pattern that dithering moves. A short sub on
  a 5-hour night means hundreds of subs, so this choice decides the short end of the curve. **Past
  N = 18 it is an assumption**, labelled where it bites.
- **A star has lost its colour if its core clips in any plane.** One curve, not four. Red clips
  most on this night, so red will mostly set this half.

**Fixed defaults.**

- **SNR is judged on blue**, the dimmest plane: MISSION says the dimmest plane sets the exposure
  floor. Green is computed beside it.
- **"Faint" means `F_obj` = 1% of the plane's sky.** The SNR here is relative, and in the faint limit
  `F_obj` cancels out of every ratio.
- **The night is this night**: its length, `t_dead` with dithering every frame, `F_sky` per plane.
- **Each star's light is from `29`**, measured in cell A at -20 C: `model.f_star_peak` turns its
  median peak into electrons per second, with `15`'s cold `g` and pedestal. It is then carried to
  each setting with the model's own constants at -10 C, which is where the model runs. The few stars
  already clipped in A have only a lower bound, and are counted as clipped at every setting.
- **The ceiling is `linear_to_at_least`, not 4095**, as MISSION says. The curve therefore reads more
  clipping than `29` measured at the same setting, which counted 4095; section 4 shows by how much.
- **The 2% that a long sub loses (`29`) is left out.** It is on the safe side.

**What it writes.** `results/pareto_curve.csv`, every setting computed, and
`results/pareto_constants.json`, the points on the curve with their provenance. These are **model
predictions**, as `17`'s `sky_pairs.csv` was: nothing here is measured. `32` reads them back.

**What it is not for.** Not other targets or nights: every star number belongs to NGC 7000, this
night's seeing and this focus. Not gains outside 50, 100 and 200. Not a recommendation. Not the
mount's limit on sub length, which is unmeasured: the curve runs to 600 s, and whether the mount
tracks that long is a separate question.

**It assumes** `24_model_constants_read.ipynb` for the model's constants and `30_star_clipping_read.ipynb`
for the stars.

In [ ]:
import json
import math
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M

RESULTS = pathlib.Path.cwd().parent / "results"
mc = json.loads((RESULTS / "model_constants.json").read_text())
cold = json.loads((RESULTS / "cold_constants.json").read_text())
sky = json.loads((RESULTS / "sky_constants.json").read_text())
peaks = pd.read_csv(RESULTS / "star_peaks.csv")
frames = pd.read_csv(RESULTS / "sky_frames.csv", parse_dates=["t"])

PLANES = ("R", "G1", "G2", "B")
G_TABLE = mc["system_gain"]["value"]
RN_TABLE = mc["read_noise_counts"]["value"]
PED_FIT = mc["pedestal_fit"]["value"]
HCG = mc["hcg_threshold_gain"]["value"]
CEIL = {int(k): v for k, v in mc["linear_to_at_least"]["value"].items()}
F_SKY = mc["F_sky"]["value"]
DARK = mc["dark_current_bound"]["value"]
T_DEAD = mc["t_dead"]["value"]
ETA = 1.0                                   # agreed: no stacking loss
F_OBJ_FRAC = 0.01                           # faint: 1% of the plane's sky
SNR_PLANE, ALSO = "B", "G1"

GAINS = sorted(CEIL)                        # 50, 100, 200: where a ceiling was proved
T_GRID = np.unique(np.concatenate([np.round(np.geomspace(10, 600, 90)), [30, 120]]))

night = frames.sort_values("t")
T_NIGHT = ((night.t.iloc[-1] - night.t.iloc[0]).total_seconds() + night.exptime.iloc[-1])
CURVE_CSV = RESULTS / "pareto_curve.csv"
CONSTANTS = RESULTS / "pareto_constants.json"
print(f"night {T_NIGHT / 3600:.2f} h, t_dead {T_DEAD:.2f} s, gains {GAINS}, "
      f"{len(T_GRID)} sub lengths {T_GRID.min():.0f}-{T_GRID.max():.0f} s")
print("ceiling (linear_to_at_least):", CEIL)

## 1. Each star's light, in electrons per second

From `29`'s median peak in cell A (gain 50, 30 s), per plane, with the constants `29` used to
measure it.

In [ ]:
G_COLD = cold["system_gain_cold_per_plane"]["value"]["50"]
PED_COLD = cold["pedestal_cold"]["value"]["50"]
F_SKY_A = sky["F_sky_per_cell"]["value"]["A"]
a = peaks[peaks.cell == "A"].pivot(index="star", columns="plane")
F = pd.DataFrame({p: M.f_star_peak(a.peak_median[p], 30.0, f_sky=F_SKY_A[p], dark=DARK,
                                   pedestal=PED_COLD, g=G_COLD[p]) for p in PLANES})
CLIPPED_IN_A = a.clipped.any(axis=1).to_numpy()
print(f"{len(F)} stars; {CLIPPED_IN_A.sum()} clipped in some plane already in A "
      f"({100 * CLIPPED_IN_A.mean():.2f}%), counted as clipped everywhere")
print(F.describe(percentiles=[0.5, 0.99, 0.999]).loc[["50%", "99%", "99.9%", "max"]].round(1))

## 2. Every setting: SNR and clipped share

For each gain and sub length: the sub count the night holds, the SNR per plane, and which stars
reach the ceiling in which plane.

In [ ]:
rows = []
for gain in GAINS:
    g = M.g_at(gain, G_TABLE)
    read_e = M.read_noise_e(gain, RN_TABLE, G_TABLE)
    ped = M.pedestal_counts(gain, PED_FIT, HCG)
    for t in T_GRID:
        n = M.subs_in(T_NIGHT, t, T_DEAD)
        row = {"gain": gain, "t": t, "n_subs": n, "beyond_N18": n > 18}
        clipped_any = CLIPPED_IN_A.copy()
        for p in PLANES:
            row[f"snr_{p}"] = ETA * math.sqrt(n) * M.snr_sub(
                F_OBJ_FRAC * F_SKY[p], t, f_sky=F_SKY[p], dark=DARK, read_e=read_e)
            peak = M.peak_counts(F[p].to_numpy(), t, f_sky=F_SKY[p], dark=DARK, pedestal=ped, g=g)
            hit = peak >= CEIL[gain]
            row[f"clipped_{p}"] = float((hit | CLIPPED_IN_A).mean())
            clipped_any |= hit
        row["clipped_any"] = float(clipped_any.mean())
        rows.append(row)
curve = pd.DataFrame(rows)
for p in PLANES:
    curve[f"snr_rel_{p}"] = curve[f"snr_{p}"] / curve[f"snr_{p}"].max()
print(curve[["gain", "t", "n_subs", f"snr_rel_{SNR_PLANE}", f"snr_rel_{ALSO}", "clipped_any"]]
      [curve.t.isin([10, 30, 60, 120, 300, 600])].round(4).to_string(index=False))

## 3. The curve

A setting is on the curve if no other setting has at least its SNR with no more clipping, and is
better on one of them. Read it as a menu: for each amount of star colour you are willing to lose,
the most faint-signal SNR the night can give.

In [ ]:
s = curve.sort_values(["clipped_any", f"snr_rel_{SNR_PLANE}"], ascending=[True, False])
best = -np.inf
front = []
for i, r in s.iterrows():
    if r[f"snr_rel_{SNR_PLANE}"] > best:
        front.append(i)
        best = r[f"snr_rel_{SNR_PLANE}"]
curve["on_curve"] = curve.index.isin(front)
f = curve[curve.on_curve].sort_values("clipped_any")
print(f"{len(f)} settings on the curve; by gain: {f.gain.value_counts().sort_index().to_dict()}")
print(f[["gain", "t", "n_subs", "beyond_N18", f"snr_rel_{SNR_PLANE}", f"snr_rel_{ALSO}",
         "clipped_any"]].round(4).to_string(index=False))

**Read at a few budgets.** For a chosen share of stars allowed to clip, the best setting and what
it buys. This is the curve read off, not a recommendation.

In [ ]:
budgets = []
for b in (0.0025, 0.005, 0.01, 0.02, 0.05):
    ok = curve[curve.clipped_any <= b]
    if ok.empty:
        budgets.append({"budget_pct": 100 * b, "gain": None})
        continue
    r = ok.loc[ok[f"snr_rel_{SNR_PLANE}"].idxmax()]
    budgets.append({"budget_pct": 100 * b, "gain": int(r.gain), "t": r.t, "n_subs": int(r.n_subs),
                    "beyond_N18": bool(r.beyond_N18), f"snr_rel_{SNR_PLANE}": r[f"snr_rel_{SNR_PLANE}"],
                    "clipped_any_pct": 100 * r.clipped_any})
budgets = pd.DataFrame(budgets)
print(budgets.round(4).to_string(index=False))

## 4. Where the night's four cells sit

The four settings actually shot, read off the curve's table, beside the clipped share `29`
measured there. The curve counts a star as clipped at `linear_to_at_least`; `29` counted 4095. The
gap is the safety margin MISSION chose, not an error.

In [ ]:
CELLS = {"A": (50, 30.0), "B": (50, 120.0), "C": (200, 30.0), "D": (200, 120.0)}
measured_any = peaks.groupby(["cell", "star"]).clipped.any().groupby("cell").mean()
cells = pd.DataFrame([{**{"cell": c, "gain": g, "t": t},
                       **curve[(curve.gain == g) & (curve.t == t)]
                       [[f"snr_rel_{SNR_PLANE}", "clipped_any", "on_curve"]].iloc[0].to_dict(),
                       "clipped_any_at_4095_measured": measured_any[c]}
                      for c, (g, t) in CELLS.items()])
print(cells.round(4).to_string(index=False))

## 5. Publish

In [ ]:
measured_on = str(night.t.iloc[0].date())
curve.round(6).to_csv(CURVE_CSV, index=False)
inputs = (f"model predictions, nothing measured.  SNR of faint signal (F_obj = {F_OBJ_FRAC:.0%} of "
          f"sky) in plane {SNR_PLANE}, relative to the best setting, over a night of "
          f"{T_NIGHT / 3600:.2f} h with t_dead {T_DEAD:.2f} s and eta_comb = {ETA} (agreed; "
          f"measured only to N = 18 -- beyond_N18 marks where it is assumed).  Clipping: the "
          f"share of 29's {len(F)} isolated stars whose core reaches linear_to_at_least in any "
          f"plane, each star's F_star_peak from its median peak in cell A; the "
          f"{int(CLIPPED_IN_A.sum())} stars clipped in A are counted clipped everywhere.  Model "
          f"constants at -10 C: g_at, read_noise_e, pedestal_counts, F_sky per plane.  Gains "
          f"{GAINS} only, where a ceiling was proved.  NGC 7000, this night's seeing and focus")
constants = {
    "pareto_front": {
        "value": [{"gain": int(r.gain), "t": float(r.t), "n_subs": int(r.n_subs),
                   "snr_rel": round(float(r[f"snr_rel_{SNR_PLANE}"]), 5),
                   "clipped_any": round(float(r.clipped_any), 6)}
                  for _, r in f.iterrows()],
        "unit": "settings on the curve: relative SNR (plane B) and share of stars clipped",
        "uncertainty": None, "source_frames": int(len(frames)), "measured_on": measured_on,
        "notebook": "31_tradeoff_curve.ipynb", "note": inputs},
}
with open(CONSTANTS, "w") as fh:
    json.dump(constants, fh, indent=2)
M.Constants.load(CONSTANTS)       # the provenance gate
print("wrote", CURVE_CSV.name, f"({len(curve)} settings),", CONSTANTS.name, f"({len(f)} on the curve)")